# 🦟 ระบบปัญญาประดิษฐ์เฝ้าระวังชนิดลูกน้ำยุงและพาหะนำโรค (Mosquito Larvae AI Surveillance)
### โครงการวิจัยและพัฒนา AIoT เพื่อการเกษตรแม่นยำและสุขภาพหนึ่งเดียว (One Health AIoT Initiative)
**คณะวิทยาศาสตร์และเทคโนโลยี มหาวิทยาลัยราชภัฏรำไพพรรณี (SCIRBRU)**  
**พัฒนาโดย:** ผศ.ดร.ชีวะ ทัศนา และคณะ  
**เทคโนโลยี:** Ultralytics YOLO11 / YOLO26, Roboflow Cloud Universe, ONNX Runtime, TFLite INT8  

---
### 🔬 ข้อมูลอนุกรมวิธานลูกน้ำยุง 3 ชนิดพันธุ์สำคัญ (Target Classes)
1. 🔴 **`Aedes_aegypti`** (ยุงลายบ้าน) — พาหะนำโรคไข้เลือดออก (Dengue), ไวรัสซิกา (Zika), และไข้ปวดข้อยุงลาย (Chikungunya)
2. 🟢 **`Aedes_albopictus`** (ยุงลายสวน) — พาหะนำโรคไข้เลือดออกและไวรัสเขตร้อนในพื้นที่สวนผลไม้และพื้นที่เกษตร
3. 🟣 **`Culex_quinque`** (*Culex quinquefasciatus* ยุงรำคาญ) — พาหะนำโรคพยาธิเท้าช้าง (Filariasis) และไข้สมองอักเสบเจอี (JE)

---
### ⚡ การตั้งค่าสภาพแวดล้อม (Hardware Acceleration)
> **คำแนะนำ:** ก่อนเริ่มรัน ให้ไปที่เมนู **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ เลือก **T4 GPU**

## 1. ติดตั้งไลบรารีและตรวจสอบการทำงานของชิปเร่งความเร็ว GPU

In [ ]:
# ติดตั้ง Ultralytics, Supervision และ Roboflow
%pip install -q "ultralytics==8.3.40" supervision roboflow

import ultralytics
import torch

# ตรวจสอบระบบและ GPU
ultralytics.checks()

if torch.cuda.is_available():
    print(f"\n🚀 เชื่อมต่อ GPU สำเร็จ: {torch.cuda.get_device_name(0)}")
    print(f"💾 หน่วยความจำ VRAM รวม: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ ยังไม่ได้เปิดใช้งาน GPU (กำลังรันบน CPU)")

## 2. ดาวน์โหลดชุดข้อมูลลูกน้ำยุง (Mosquito Larvae v3) จาก Roboflow

ดึงข้อมูล Version 3 ฟอร์แมต `yolo26` จำนวน **2,730 ภาพ** ที่ผ่านการทำ Data Augmentation เรียบร้อยแล้ว

In [ ]:
from roboflow import Roboflow

# เชื่อมต่อ Roboflow API
rf = Roboflow(api_key="tdetHe9WG2CTItfj0VHt")
project = rf.workspace("durian-nodisease").project("mosquito_larvae-ivnri")
version = project.version(3)

# ดาวน์โหลดชุดข้อมูลฟอร์แมต yolo26
dataset = version.download("yolo26")

print("\n" + "="*60)
print(f"✅ ดาวน์โหลดชุดข้อมูลลูกน้ำยุงสำเร็จ!")
print(f"📁 ตำแหน่งไฟล์ข้อมูล: {dataset.location}")
print("="*60)

## 3. สำรวจโครงสร้างชุดข้อมูลและตรวจสอบไฟล์ `data.yaml`

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path(dataset.location) / "data.yaml"
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📋 ข้อมูลการกำหนดค่าชุดข้อมูล (Dataset Configuration):")
print(f"- จำนวนคลาส (nc): {data_config.get('nc')}")
print(f"- รายชื่อคลาส (names): {data_config.get('names')}")
print(f"- สัดส่วน Train: {data_config.get('train')}")
print(f"- สัดส่วน Val: {data_config.get('val')}")

## 4. เริ่มต้นการฝึกสอนโมเดลตรวจจับลูกน้ำยุงด้วย YOLO11

ใช้สถาปัตยกรรม **YOLO11 Nano (`yolo11n.pt`)** ซึ่งมีความเร็วในการประมวลผลสูงมากและใช้พลังงานต่ำ เหมาะสำหรับบอร์ด Edge AI และกล้องดักจับลูกน้ำยุงอัตโนมัติ

In [ ]:
from ultralytics import YOLO

# โหลดพรีเทรนโมเดล YOLO11 Nano
model = YOLO("yolo11n.pt")

# เริ่มการฝึกสอนโมเดล
results = model.train(
    data=str(yaml_path),
    epochs=50,          # จำนวนรอบการเทรน
    imgsz=640,          # ขนาดภาพนำเข้า 640x640 พิกเซล
    batch=16,           # ขนาด Batch สำหรับ GPU T4
    device=0,           # รันบน GPU 0
    plots=True,         # สร้างกราฟสรุปผล
    name="mosquito_larvae_yolo11"
)

## 5. แสดงผลกราฟความแม่นยำ (mAP50, Precision, Recall, Loss Curves)

In [ ]:
from IPython.display import Image, display
import os

result_img = "runs/detect/mosquito_larvae_yolo11/results.png"
if os.path.exists(result_img):
    print("📊 กราฟประวัติการฝึกสอนโมเดล YOLO11:")
    display(Image(result_img, width=800))
else:
    print("ไม่พบไฟล์ภาพผลลัพธ์")

## 6. วิเคราะห์ประสิทธิภาพบนชุดทดสอบและแผนภาพความสับสน (Confusion Matrix)

In [ ]:
# ประเมินผลความแม่นยำอย่างเป็นทางการ
metrics = model.val()

print("\n" + "="*50)
print(f"🎯 ค่าความแม่นยำเฉลี่ย mAP@0.50: {metrics.box.map50 * 100:.2f}%")
print(f"🎯 ค่าความแม่นยำรวม mAP@0.50-0.95: {metrics.box.map * 100:.2f}%")
print("="*50)

# แสดงแผนภาพความสับสน (Confusion Matrix)
cm_img = "runs/detect/mosquito_larvae_yolo11/confusion_matrix.png"
if os.path.exists(cm_img):
    print("\n🧩 แผนภาพความสับสน (Normalized Confusion Matrix):")
    display(Image(cm_img, width=650))

## 7. ทดสอบการทำนายภาพตัวอย่างลูกน้ำยุง (Live Inference Playground)

In [ ]:
import glob

# ค้นหาภาพตัวอย่างจาก Validation / Test set
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg") + glob.glob(f"{dataset.location}/valid/images/*.jpg")

if test_images:
    sample_test_img = test_images[0]
    print(f"🔍 กำลังทดสอบกับภาพ: {sample_test_img}")
    
    # รันโมเดลทำนายผล
    preds = model.predict(source=sample_test_img, conf=0.25, save=True, project="runs/detect", name="inference_test")
    
    # แสดงภาพผลลัพธ์พร้อม Bounding Box และชื่อชนิดยุง
    pred_img = f"runs/detect/inference_test/{os.path.basename(sample_test_img)}"
    if os.path.exists(pred_img):
        display(Image(pred_img, width=600))
else:
    print("ไม่พบภาพตัวอย่างสำหรับทดสอบ")

## 8. การส่งออกโมเดล (Export) สู่มาตรฐานสากลสำหรับติดตั้งบน Edge AI & IoT

1. **ONNX (Open Neural Network Exchange):** สำหรับรันบน Raspberry Pi 4/5, คอมพิวเตอร์เกตเวย์ และแอปพลิเคชัน
2. **TensorFlow Lite INT8:** บีบอัดขนาดไฟล์เหลือเพียง ~2.8 MB สำหรับกล้องตรวจการณ์บ่อน้ำขัง ESP32-S3

In [ ]:
# ส่งออกเป็น ONNX
onnx_path = model.export(format="onnx", imgsz=640)
print(f"✅ ส่งออกโมเดล ONNX สำเร็จ: {onnx_path}")

# ส่งออกเป็น TensorFlow Lite INT8 (Quantized)
tflite_path = model.export(format="tflite", imgsz=320, int8=True)
print(f"✅ ส่งออกโมเดล TFLite INT8 สำเร็จ: {tflite_path}")

## 9. ดาวน์โหลดไฟล์โมเดลสำเร็จรูปสู่เครื่องคอมพิวเตอร์ของคุณ

In [ ]:
import glob
from google.colab import files

print("📦 รายการไฟล์โมเดลตรวจจับลูกน้ำยุงที่พร้อมนำไปใช้งาน:")
for f in glob.glob("runs/detect/mosquito_larvae_yolo11/weights/best*"):
    sz_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  - {f} (ขนาด: {sz_mb:.2f} MB)")

# หากต้องการดาวน์โหลดไฟล์โมเดลลงเครื่องคอมพิวเตอร์ ให้ปลด comment บรรทัดด้านล่าง:
# files.download('runs/detect/mosquito_larvae_yolo11/weights/best.onnx')
# files.download('runs/detect/mosquito_larvae_yolo11/weights/best_int8.tflite')